# 01 · Exploratory Data Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Cecile1015/emotion-mirror-cbs4702/blob/main/notebooks/01_eda.ipynb)

**Dataset:** [dair-ai/emotion](https://huggingface.co/datasets/dair-ai/emotion) (Saravia et al., 2018) — 20,000 English texts labelled with one of six emotions: *sadness (0), joy (1), love (2), anger (3), fear (4), surprise (5)*.

This notebook characterises the data before any model is trained. Each analysis ends with a short note on how the finding affects model development.

| § | Analysis | Question |
|---|---|---|
| 1 | Loading and integrity checks | Are the splits complete and well-formed? |
| 2 | Label distribution | How imbalanced are the classes, and are the splits comparable? |
| 3 | Text length | How long are the texts, and does length differ by emotion? |
| 4 | Vocabulary and Zipf's law | How sparse is the vocabulary, and how many unseen words occur at test time? |
| 5 | Frequent vs. distinctive words | Which words characterise each emotion? |
| 6 | Lexical templates | Does the data-collection method leave a trace in the texts? |
| 7 | Duplicates and split overlap | Is there label noise or leakage between splits? |
| 8 | Difficult examples | Which texts and which emotion pairs are intrinsically ambiguous? |
| 9 | Evaluation sanity check | What score does a trivial classifier obtain? |
| 10 | Summary | What do the findings imply for the models? |

Test **labels** are never used for any modelling decision. Test texts appear only in descriptive comparisons (label shares, vocabulary coverage, overlap counts).

In [ ]:
# Environment setup: runs locally (from the project root or notebooks/) and on Google Colab.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Cecile1015/emotion-mirror-cbs4702.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Colab already provides every package this notebook needs; only the repository is cloned.
    ROOT = Path("/content/emotion-mirror-cbs4702")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    os.chdir(ROOT / "notebooks")
else:
    ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

sys.path.insert(0, str(ROOT))
print(f"Project root: {ROOT} | running on Colab: {IN_COLAB}")

In [ ]:
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.stats import chi2_contingency, kruskal
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline

from src.config import EMOTION_COLORS, FIG_DIR, LABELS, SEED, ensure_dirs, set_seed
from src.data import load_splits, tokenize
from src.evaluate import emotion_palette, evaluate, per_class_table, plot_confusion_matrix

set_seed(SEED)
ensure_dirs()
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "axes.titleweight": "bold"})
PALETTE = emotion_palette()
pd.set_option("display.max_colwidth", 120)


def savefig(fig, name):
    fig.savefig(FIG_DIR / name, bbox_inches="tight")

## 1. Loading and integrity checks

The three official splits are downloaded once and cached in `data/` (see `src/data.py`). `load_split` also validates the schema: no missing texts and all labels within 0–5.

In [ ]:
splits = load_splits()
train, val, test = splits["train"], splits["validation"], splits["test"]

for name, df in splits.items():
    n_empty = int((df["text"].str.strip() == "").sum())
    print(f"{name:<11} {len(df):>6,} rows | empty texts: {n_empty} | label ids: {sorted(int(x) for x in df['label'].unique())}")

train.sample(8, random_state=SEED)[["text", "label", "label_name"]]

## 2. Label distribution

The class balance determines the evaluation metric and whether class weighting is needed. A χ² test of homogeneity checks whether the three splits share the same label distribution.

In [ ]:
dist = pd.DataFrame({name: df["label_name"].value_counts().reindex(LABELS) for name, df in splits.items()})
share = dist / dist.sum()
display(pd.concat({"count": dist, "share (%)": (share * 100).round(1)}, axis=1))

imbalance_ratio = dist["train"].max() / dist["train"].min()
chi2_stat, chi2_p, _, _ = chi2_contingency(dist.values)
print(f"Imbalance ratio (largest / smallest class, train): {imbalance_ratio:.1f}")
print(f"Chi-squared test of homogeneity across splits: chi2 = {chi2_stat:.2f}, p = {chi2_p:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
ax = axes[0]
bars = ax.barh(LABELS[::-1], dist["train"][::-1], color=PALETTE[::-1], edgecolor="black", linewidth=0.4)
for bar, n, s in zip(bars, dist["train"][::-1], share["train"][::-1]):
    ax.text(bar.get_width() + 60, bar.get_y() + bar.get_height() / 2, f"{n:,} ({s:.1%})", va="center", fontsize=9)
ax.set_xlim(0, dist["train"].max() * 1.25)
ax.set_xlabel("Number of texts")
ax.set_title("Training set: texts per emotion")

ax = axes[1]
x = np.arange(len(LABELS))
for k, (name, shade) in enumerate(zip(splits, ["#2F2F2F", "#8C8C8C", "#C8C8C8"])):
    ax.bar(x + (k - 1) * 0.27, share[name] * 100, 0.27, label=name, color=shade, edgecolor="black", linewidth=0.3)
ax.set_xticks(x, LABELS)
ax.set_ylabel("Share of split (%)")
ax.set_title("Label shares are consistent across splits")
ax.legend()
fig.tight_layout()
savefig(fig, "eda_label_distribution.png")
plt.show()

**Implication.** The classes are clearly imbalanced: *joy* and *sadness* dominate while *surprise* and *love* are minority classes. Accuracy alone would reward a model that neglects the minority classes, so **macro-averaged F1** is used as the primary metric for model selection, with accuracy and per-class scores reported alongside. Class weighting is tested as a controlled experiment in the improved models. Because the splits have matching label shares, validation scores should be a reliable guide to test performance.

## 3. Text length

Length is measured in word tokens (`src.data.tokenize`). Length matters for two reasons: very short texts give a bag-of-words model little evidence, and the length percentiles set the maximum sequence length for the LSTM and BERT models.

In [ ]:
for df in splits.values():
    df["n_tokens"] = df["text"].map(lambda t: len(tokenize(t)))

length_stats = pd.DataFrame({name: df["n_tokens"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]) for name, df in splits.items()}).T
display(length_stats.round(1))

by_class = train.groupby("label_name")["n_tokens"].agg(["median", "mean", "std"]).reindex(LABELS)
display(by_class.round(1))

h_stat, h_p = kruskal(*[train.loc[train["label_name"] == c, "n_tokens"] for c in LABELS])
print(f"Kruskal-Wallis test of length differences between emotions: H = {h_stat:.1f}, p = {h_p:.2g}")

p95, p99 = np.percentile(train["n_tokens"], [95, 99])
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
ax = axes[0]
ax.hist(train["n_tokens"], bins=range(0, int(train["n_tokens"].max()) + 2), color="#7A7A7A", edgecolor="white")
for value, style, label in [(p95, "--", "95th percentile"), (p99, ":", "99th percentile")]:
    ax.axvline(value, color="black", linestyle=style, label=f"{label} = {value:.0f}")
ax.set_xlabel("Tokens per text")
ax.set_ylabel("Number of texts")
ax.set_title("Training set: text length")
ax.legend(loc="upper right")

ax = axes[1]
sns.boxplot(data=train, x="label_name", y="n_tokens", order=LABELS, hue="label_name",
            palette=EMOTION_COLORS, legend=False, fliersize=1.5, ax=ax)
ax.set_xlabel("")
ax.set_ylabel("Tokens per text")
ax.set_title("Text length by emotion")
fig.tight_layout()
savefig(fig, "eda_text_length.png")
plt.show()

**Implication.** The texts are short single sentences, so each prediction rests on a small amount of lexical evidence; one emotionally loaded word can decide the label. Length differences between emotions are statistically detectable but small in practical terms, so length is not a useful feature on its own. The 99th percentile is used to choose the maximum sequence length for the neural models, which avoids truncating almost any text while keeping padding low.

## 4. Vocabulary, Zipf's law and unseen words

Word frequencies in natural language follow Zipf's law: frequency is roughly inversely proportional to frequency rank, so most word types are rare. This is the sparsity problem that motivates smoothing in n-gram language models (Lecture 5) and dense or sub-word representations (Lecture 6).

In [ ]:
train_counts = Counter(tok for text in train["text"] for tok in tokenize(text))
vocab = set(train_counts)
n_hapax = sum(1 for c in train_counts.values() if c == 1)
print(f"Training vocabulary: {len(vocab):,} word types, {sum(train_counts.values()):,} tokens")
print(f"Hapax legomena (types seen once): {n_hapax:,} ({n_hapax / len(vocab):.1%} of types)")

oov_rows = []
for name in ["validation", "test"]:
    toks = [tokenize(t) for t in splits[name]["text"]]
    n_tok = sum(len(t) for t in toks)
    n_oov = sum(1 for t in toks for w in t if w not in vocab)
    has_oov = np.mean([any(w not in vocab for w in t) for t in toks])
    oov_rows.append({"split": name, "OOV token rate": n_oov / n_tok, "texts with >= 1 OOV word": has_oov})
display(pd.DataFrame(oov_rows).set_index("split").style.format("{:.1%}"))

freqs = np.array(sorted(train_counts.values(), reverse=True))
ranks = np.arange(1, len(freqs) + 1)
fit_range = slice(9, min(2000, len(freqs)))
slope, intercept = np.polyfit(np.log10(ranks[fit_range]), np.log10(freqs[fit_range]), 1)

fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.loglog(ranks, freqs, ".", markersize=2, color="#3B6BA5", label="observed")
ax.loglog(ranks[fit_range], 10 ** (intercept + slope * np.log10(ranks[fit_range])), "k--", label=f"fit, slope = {slope:.2f}")
ax.set_xlabel("Frequency rank (log)")
ax.set_ylabel("Frequency (log)")
ax.set_title("Zipf plot of the training vocabulary")
ax.legend()
savefig(fig, "eda_zipf.png")
plt.show()

**Implication.** A large share of word types occurs only once, and a noticeable proportion of validation and test texts contain at least one word never seen in training. A word-level bag-of-words model has no information about such words. Character n-gram features — the same idea as the sub-word units in FastText — and the WordPiece tokenizer of BERT share information across related word forms (*annoy, annoyed, annoying*), which motivates them as improved representations.

## 5. Frequent versus distinctive words

Raw frequency mostly surfaces function words that occur in every class. To find words that *distinguish* an emotion, each class is compared with all other classes using the **weighted log-odds ratio with an informative Dirichlet prior** (Monroe, Colaresi & Quinn, 2008). The z-score accounts for the uncertainty of rare words, so it does not over-rank words that appear only a handful of times.

In [ ]:
class_counts = {c: Counter(tok for text in train.loc[train["label_name"] == c, "text"] for tok in tokenize(text)) for c in LABELS}

top_frequent = pd.DataFrame({c: [w for w, _ in class_counts[c].most_common(12)] for c in LABELS})
stop_share = top_frequent.isin(ENGLISH_STOP_WORDS).mean()
print("Share of stop words among the 12 most frequent words per class:")
print(stop_share.round(2).to_string())
top_frequent

In [ ]:
def log_odds_dirichlet(counts_i, counts_j, prior, alpha0=1000):
    # Weighted log-odds ratio (class i vs. class j) with an informative Dirichlet prior
    # proportional to overall word frequency (Monroe et al., 2008, eq. 22).
    n_i, n_j, n_prior = sum(counts_i.values()), sum(counts_j.values()), sum(prior.values())
    scores = {}
    for w, p in prior.items():
        a_w = alpha0 * p / n_prior
        y_i, y_j = counts_i.get(w, 0), counts_j.get(w, 0)
        delta = np.log((y_i + a_w) / (n_i + alpha0 - y_i - a_w)) - np.log((y_j + a_w) / (n_j + alpha0 - y_j - a_w))
        scores[w] = delta / np.sqrt(1 / (y_i + a_w) + 1 / (y_j + a_w))
    return pd.Series(scores)


MIN_COUNT = 10
prior = Counter({w: c for w, c in train_counts.items() if c >= MIN_COUNT})
distinctive = {}
for c in LABELS:
    rest = Counter()
    for other in LABELS:
        if other != c:
            rest.update(class_counts[other])
    distinctive[c] = log_odds_dirichlet(class_counts[c], rest, prior).sort_values(ascending=False)

fig, axes = plt.subplots(2, 3, figsize=(14, 7.5))
for ax, c in zip(axes.flat, LABELS):
    top = distinctive[c].head(10)[::-1]
    ax.barh(top.index, top.values, color=EMOTION_COLORS[c])
    ax.set_title(c)
    ax.set_xlabel("z-score")
fig.suptitle("Most distinctive words per emotion (weighted log-odds, informative Dirichlet prior)", fontweight="bold")
fig.tight_layout()
savefig(fig, "eda_distinctive_words.png")
plt.show()

pd.DataFrame({c: distinctive[c].head(10).index for c in LABELS})

**Implication.** The frequency lists are dominated by function words and by *feel*, which carry little class information; this motivates TF-IDF weighting (Lecture 6), which down-weights terms that occur in many documents. The distinctive words, by contrast, are mostly emotion adjectives, which explains why linear models over sparse word features are strong baselines on this dataset. The lists also show which pairs of classes share vocabulary (for example affectionate words that appear in both *love* and *joy*), anticipating the main confusions in §8.

## 6. Lexical templates left by data collection

The CARER corpus was collected from social-media posts using emotion-related queries and hashtags (Saravia et al., 2018). A trace of that procedure would appear as a recurring sentence template. The table counts texts containing a form of *feel* and the most common sentence openings, and extracts the word that follows *feel* (skipping intensifiers such as *so* or *really*).

In [ ]:
FEEL_RE = re.compile(r"\b(?:feel|feels|feeling|felt)\b")
INTENSIFIERS = r"(?:so|very|really|quite|a|bit|little|pretty|rather|too|more|less|kind|of|like|that|extremely|incredibly|super|somewhat|slightly|totally|completely|just|still|even|all|not)"
AFTER_FEEL_RE = re.compile(rf"\b(?:feel|feels|feeling|felt)\s+(?:{INTENSIFIERS}\s+)*([a-z']+)")

template = pd.DataFrame({
    "contains feel/feeling/felt": train.groupby("label_name")["text"].apply(lambda s: s.str.contains(FEEL_RE).mean()),
    "starts with 'i feel'": train.groupby("label_name")["text"].apply(lambda s: s.str.startswith("i feel").mean()),
}).reindex(LABELS)
template.loc["all classes"] = [train["text"].str.contains(FEEL_RE).mean(), train["text"].str.startswith("i feel").mean()]
display(template.style.format("{:.1%}"))

openings = Counter(" ".join(tokenize(t)[:2]) for t in train["text"])
print("Most common sentence openings:", ", ".join(f"'{o}' ({n:,})" for o, n in openings.most_common(8)))

after_feel = train.assign(word=train["text"].str.extract(AFTER_FEEL_RE, expand=False))
pd.DataFrame({c: after_feel.loc[after_feel["label_name"] == c, "word"].value_counts().head(8).index for c in LABELS})

**Implication.** A large proportion of texts follow the template *"i feel …"*, and the word that follows *feel* is often itself the emotion label (or a close synonym). The label therefore frequently hinges on a single adjective. This makes the task easier for lexical models but also creates a risk of **shortcut learning**: a model may learn adjective–label associations without modelling negation (*"i don't feel happy"*), intensity or irony. These phenomena are targeted explicitly in the error analysis.

## 7. Duplicates, label conflicts and overlap between splits

Exact duplicate texts with *different* labels are direct evidence of label noise, and texts shared between training and evaluation splits would inflate the reported scores. For the test split only the number of overlapping texts is reported; test labels are not inspected.

In [ ]:
dup_mask = train["text"].duplicated(keep=False)
conflicts = train[dup_mask].groupby("text")["label_name"].agg(lambda s: sorted(set(s)))
conflicts = conflicts[conflicts.map(len) > 1]
print(f"Training texts that occur more than once: {int(dup_mask.sum()):,} rows ({train.loc[dup_mask, 'text'].nunique():,} distinct texts)")
print(f"Duplicated texts with conflicting labels: {len(conflicts):,}")
display(conflicts.head(8).to_frame("labels"))

train_texts = set(train["text"])
overlap = pd.Series({
    "train ∩ validation": len(train_texts & set(val["text"])),
    "train ∩ test": len(train_texts & set(test["text"])),
    "validation ∩ test": len(set(val["text"]) & set(test["text"])),
}, name="shared texts")
display(overlap.to_frame())

shared_val = val[val["text"].isin(train_texts)].merge(
    train.drop_duplicates("text")[["text", "label_name"]], on="text", suffixes=("_val", "_train"))
if len(shared_val):
    agree = (shared_val["label_name_val"] == shared_val["label_name_train"]).mean()
    print(f"Label agreement on texts shared by train and validation: {agree:.1%} of {len(shared_val)}")

**Implication.** Conflicting labels on identical texts set a ceiling on achievable accuracy: no model can be right for both copies. Together with the distant-supervision origin of the labels (§6), they suggest a category of *"dataset label noise"* in the error analysis, where the model's prediction may be more defensible than the gold label. The overlap between splits is small, so it does not materially inflate the evaluation, but it is reported for transparency.

## 8. Ambiguous and difficult-to-classify texts

To locate intrinsically difficult examples without touching the evaluation splits, a simple **probe** — TF-IDF features with logistic regression — is trained with 5-fold stratified cross-validation **on the training set only**. Every training text therefore receives a prediction from a model that did not see it. Texts that receive a low probability for their own gold label are either ambiguous, express several emotions, or are mislabelled. The probe is a diagnostic tool, not one of the compared models.

In [ ]:
probe = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
    LogisticRegression(max_iter=2000, random_state=SEED),
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_proba = cross_val_predict(probe, train["text"], train["label"], cv=cv, method="predict_proba", n_jobs=-1)
train["cv_pred"] = cv_proba.argmax(axis=1)
train["gold_prob"] = cv_proba[np.arange(len(train)), train["label"]]
train["cv_pred_name"] = train["cv_pred"].map(dict(enumerate(LABELS)))

difficulty = train.groupby("label_name").agg(cv_recall=("cv_pred", lambda s: (s == train.loc[s.index, "label"]).mean()),
                                             mean_gold_prob=("gold_prob", "mean")).reindex(LABELS)
display(difficulty.round(3))

fig, ax = plt.subplots(figsize=(6.4, 5.2))
plot_confusion_matrix(train["label"], train["cv_pred"], title="Training-set probe (5-fold CV): confusable emotions", ax=ax,
                      save_as="eda_probe_confusion.png")
plt.show()

In [ ]:
likely_noise = train[(train["gold_prob"] < 0.05) & (cv_proba.max(axis=1) > 0.90)]
print(f"Confidently contradicted labels (gold prob < 0.05, probe confidence > 0.90): {len(likely_noise):,} "
      f"({len(likely_noise) / len(train):.1%} of training texts)")

hardest = train.nsmallest(12, "gold_prob")[["text", "label_name", "cv_pred_name", "gold_prob"]]
hardest.columns = ["text", "gold label", "probe prediction", "P(gold)"]
hardest.style.format({"P(gold)": "{:.3f}"})

**Implication.** The probe confirms that difficulty is concentrated in specific emotion pairs rather than spread evenly: the minority classes are frequently absorbed by semantically neighbouring majority classes (for example *love → joy* and *surprise → fear/joy*). Inspecting the lowest-probability texts shows three recurring sources of difficulty that structure the later error analysis: (i) texts that mention several emotions, (ii) emotion expressed implicitly or through negation, and (iii) gold labels that a human reader would dispute.

## 9. Sanity check of the evaluation pipeline

Before training real models, `src/evaluate.py` is checked on two trivial classifiers on the **validation** set. They also give a performance floor: any useful model must clearly exceed them, especially on macro-F1.

In [ ]:
X_train_dummy = np.zeros((len(train), 1))
X_val_dummy = np.zeros((len(val), 1))
for strategy in ["most_frequent", "stratified"]:
    dummy = DummyClassifier(strategy=strategy, random_state=SEED).fit(X_train_dummy, train["label"])
    record = evaluate(f"dummy_{strategy}", val["label"], dummy.predict(X_val_dummy), split="validation",
                      texts=val["text"], notes=f"DummyClassifier(strategy='{strategy}'); sanity check")
per_class_table(record).round(3)

## 10. Summary of findings and implications for modelling

In [ ]:
summary = {
    "Training texts": f"{len(train):,}",
    "Largest / smallest class (train)": f"{dist['train'].idxmax()} {share['train'].max():.1%} / {dist['train'].idxmin()} {share['train'].min():.1%}",
    "Imbalance ratio": f"{imbalance_ratio:.1f}",
    "Median length / 99th percentile (tokens)": f"{train['n_tokens'].median():.0f} / {p99:.0f}",
    "Training vocabulary (types)": f"{len(vocab):,}",
    "Hapax share of types": f"{n_hapax / len(vocab):.1%}",
    "Test texts with >= 1 unseen word": f"{oov_rows[1]['texts with >= 1 OOV word']:.1%}",
    "Texts containing a form of 'feel'": f"{template.loc['all classes', 'contains feel/feeling/felt']:.1%}",
    "Duplicated texts with conflicting labels": f"{len(conflicts):,}",
    "Probe: confidently contradicted labels": f"{len(likely_noise) / len(train):.1%}",
    "Probe: hardest class (lowest CV recall)": f"{difficulty['cv_recall'].idxmin()} ({difficulty['cv_recall'].min():.1%})",
}
pd.Series(summary, name="value").to_frame()

| Finding | Implication for model development |
|---|---|
| Strong class imbalance; minority classes *love* and *surprise* | Macro-F1 as the primary selection metric; class weighting tested as a controlled experiment; per-class scores always reported. |
| Short single-sentence texts | Little context per prediction; a maximum length at the 99th percentile is sufficient for sequence models. |
| Zipfian vocabulary with many hapaxes and unseen test words | Character n-grams (FastText-style sub-words) and WordPiece tokenisation are expected to help over word-only features. |
| Function words dominate raw frequencies; emotion adjectives are the distinctive features | TF-IDF weighting over plain counts; linear models over sparse features are strong baselines. |
| *"i feel …"* template inherited from data collection | Labels often hinge on one adjective → risk of shortcut learning; negation, intensity and irony are expected weaknesses. |
| Shared vocabulary between *love/joy* and *fear/surprise* | Expect these pairs to dominate the confusion matrices. |
| Duplicates with conflicting labels; confidently contradicted labels | Label noise limits the achievable score and is analysed as a separate error category. |